# **🤖 TherapyAI: AI-Powered Mental Health Companion**

# 1. Introduction

## 🧠 Problem

Mental health support is increasingly vital, yet remains out of reach for millions:

- **1 in 8 people** globally live with a mental health condition (WHO, 2022)
- **85%** of people in LMICs receive **no treatment**
- Even in high-income countries, **35–50%** go untreated
- Countries spend a **global median of 2.1%** of health budgets on mental health, most of it in psychiatric institutions

Stigma, underfunding, and lack of accessible frontline care widen the gap.


## 💡 Solution

**TheraBot** offers an ethical, AI-powered support layer to:
- Detect emotional tone from audio inputs
- Generate empathetic, context-aware responses
- Retrieve practical mental wellness tips
- Responsibly escalate repeated distress with helpline prompts

It’s not therapy—but a governed companion that listens and supports instantly.


## 🎯 Goal

To create a **safe, supportive, and scalable mental health companion** using generative AI, emotion recognition, and ethical data governance.

TheraBot fills the “first-mile gap” between suffering and support.


## 🧑 Author

**Kishan Patel**  
[Kaggle: Kishan-Patel](https://www.kaggle.com/kishanpatelai) &nbsp;|&nbsp; [LinkedIn: Kishan-Patel](https://www.linkedin.com/in/kishan-patel-dev/) &nbsp;|&nbsp; [GitHub: Kishan-Patel](https://github.com/Kishan-Patel-dev/)




## 📌 Justification

Mental health conditions are widespread, under-treated, and stigmatized.

📉 Lack of access to care  
📉 Disconnected and outdated systems  
📉 Scarcity of real-time support  
📉 No governance guardrails in many AI wellness tools

TheraBot addresses this through:
- **AI + Audio + Empathy + Escalation**
- Governed logic to avoid harm
- Trustworthy and ethical design from the ground up


## 🔍 Scope

TheraBot includes:
- 🎧 Audio Emotion Detection (Speech Recognition, Few Shot Prompting)
- 💬 Prompting using Gemini + Few-Shot Examples
- 📚 Retrieval Augmented Generation (RAG) from 200+ curated tips
- 🧩 Agent logic to manage distress and trigger escalation
- 🔐 Governance layer: No harmful or clinical outputs

**Not included**:
- Clinical diagnosis  
- Medical prescriptions  
- Long-form therapy simulation


## 🔐 Governance Snapshot

- Curated dataset from public health sources (e.g., WHO, NIH)
- Escalation after 3 signals of repeated distress
- Responses reviewed and filtered for harm or bias
- Disclaimers presented in every interaction

> “Mental health care is scarce. TheraBot listens, comforts, and guides—responsibly.”

# 2. Setup

We configure tools for audio processing, generative AI, and vector search to ensure a robust pipeline.

**Goal**: To setup a error-free environment for the capstone project.

In [2]:
# Install dependencies
!pip install SpeechRecognition google-generativeai sentence-transformers faiss-cpu pydub

# Import libraries
import os
import speech_recognition as sr
import google.generativeai as genai
from pydub import AudioSegment
from sentence_transformers import SentenceTransformer
from kaggle_secrets import UserSecretsClient
import faiss
import tempfile
import numpy as np
import pandas as pd
from google.api_core import retry

## Setting API Key

In [4]:
user_secrets = UserSecretsClient()
GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

In [5]:
# Define retry policy
def is_retriable(e):
    return isinstance(e, Exception) and hasattr(e, 'code') and e.code in {429, 503}

@retry.Retry(predicate=is_retriable)
def generate_content_with_retry(model, prompt, **kwargs):
    return model.generate_content(prompt, **kwargs)

# 3. Audio Understanding: Emotion Detection

Applying Day 3’s multimodal lessons, we transcribe user audio and use Gemini to detect emotions (e.g., Stress, Sad, Anxious) and sentiment (Positive/Negative). This powers voice-driven personalization.

**Output**: 
> Transcript: I’m feeling overwhelmed
> 
> Category: Stressed, Sentiment: Negative

**Goal**: Enable seamless audio input for real-time support.

In [6]:
def transcribe_audio(audio_file):
    """Transcribe audio using speech_recognition (Day 3: Multimodal)."""
    recognizer = sr.Recognizer()
    
    if not os.path.exists(audio_file):
        return "Error: Audio file not found."
    
    try:
        with sr.AudioFile(audio_file) as source:
            audio_data = recognizer.record(source)
        text = recognizer.recognize_google(audio_data)
        return text
    except ValueError as ve:
        return f"Transcription failed: Invalid audio format ({str(ve)})."
    except sr.UnknownValueError:
        return "Transcription failed: Unclear audio."
    except sr.RequestError as re:
        return f"API error: {str(re)}."
    except Exception as e:
        return f"Unexpected error: {str(e)}."

def detect_emotion(transcript):
    """Use Gemini to detect emotion and sentiment (Day 1: Prompting)."""
    try:
        if not GOOGLE_API_KEY:  # Use correct variable
            return "Unknown", "Neutral", "Gemini API key not configured."
        
        model = genai.GenerativeModel('gemini-1.5-pro')
        prompt = """
        Examples:
        1. Text: "I feel so empty today" → Emotion: Sad, Sentiment: Negative
        2. Text: "I can’t keep up anymore" → Emotion: Stressed, Sentiment: Negative
        3. Text: "What if I mess this up?" → Emotion: Anxious, Sentiment: Negative
        4. Text: "There’s too much to do" → Emotion: Overwhelmed, Sentiment: Negative
        5. Text: "I’m so mad right now!" → Emotion: Angry, Sentiment: Negative
        6. Text: "I think tomorrow will shine" → Emotion: Hopeful, Sentiment: Positive
        7. Text: "I don’t understand what’s happening" → Emotion: Confused, Sentiment: Negative
        8. Text: "This just isn’t working out!" → Emotion: Frustrated, Sentiment: Negative
        9. Text: "I’m so thankful for today" → Emotion: Grateful, Sentiment: Positive
        10. Text: "I’m completely worn out" → Emotion: Exhausted, Sentiment: Negative
        Analyze: "{}"
        Return only: Emotion: X, Sentiment: Y
        """
        # Skip error messages
        if "failed" in transcript.lower() or "error" in transcript.lower():
            return "Unknown", "Neutral", "Input is an error message."
        
        response = generate_content_with_retry(model, prompt.format(transcript))
        if not response.text:
            return "Unknown", "Neutral", "Empty response from Gemini."
        lines = response.text.strip().split(", ")
        emotion = lines[0].replace("Emotion: ", "").strip()
        sentiment = lines[1].replace("Sentiment: ", "").strip()
        return emotion, sentiment, None
    except Exception as e:
        return "Unknown", "Neutral", f"Gemini error: {str(e)}"

## Demo for Viewers

In [7]:
audio_file = "/kaggle/input/emotions-speech/stress_1.wav"
transcript = transcribe_audio(audio_file)
emotion, sentiment, error = detect_emotion(transcript)
if error:
    print(f"Error: {error}")
print(f"Transcript: {transcript}\nCategory: {emotion}, Sentiment: {sentiment}")

Transcript: I can't keep up anymore
Category: Stressed, Sentiment: Negative


# 4. Retrieval Augmented Generation: Resource Fetching

From Day 2, RAG retrieves precise tips from our 200-row dataset (“Tip, Category, Emotions”), matching user transcripts and detected emotions for credibility.

**Output**:
> Tip: Take slow, deep breaths to calm your mind.

**Goal**: Provide expert-sourced, relevant resources.

In [8]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import json

# Load dataset
mental_health_data = pd.read_csv("/kaggle/input/mentalhealthtips200/mental_health_tips_200.csv")
data_json = mental_health_data.to_dict(orient='records')

# Format texts for embedding
texts = [
    f"emotion: {item['emotion']} | category: {item['category']} | tip: {item['tip']}"
    for item in data_json
]

# Initialize embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts, convert_to_numpy=True)
dimension = embeddings.shape[1]  # 384 for all-MiniLM-L6-v2
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Save model and index for reuse
model.save("emotion_embedding_model")
faiss.write_index(index, "emotion_index.faiss")

# Save JSON for reference
with open("MHTips_data.json", "w") as f:
    json.dump(data_json, f, indent=2)

def retrieve_tip(emotion):
    """Retrieve a tip using RAG based on detected emotion."""
    query = f"emotion: {emotion}"
    query_embedding = model.encode([query])[0]
    D, I = index.search(np.array([query_embedding]).astype(np.float32), k=1)
    return data_json[I[0][0]]['tip']

# Test RAG standalone
test_emotions = ["Stressed", "Sad", "Anxious"]
for emotion in test_emotions:
    tip = retrieve_tip(emotion)
    print(f"Emotion: {emotion}\nTip: {tip}\n")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Emotion: Stressed
Tip: Describe a view outside.



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Emotion: Sad
Tip: Allow yourself to feel and cry.



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Emotion: Anxious
Tip: List fears, then strengths.



# 5. Few-Shot Prompting: Response Generation

Applying Day 1’s prompting skills, Gemini crafts empathetic responses by merging RAG-retrieved tips with emotional context, ensuring warmth and practicality.

**Output**: 
> Response: I hear you—it’s tough to feel stressed. Take slow, deep breaths to calm your mind.

**Goal**: Deliver responses that feel human and supportive.

In [9]:
def generate_empathetic_response_fewshot(transcript, emotion, tip):
    """Generate empathetic response using Gemini with few-shot prompting, max ~2 lines."""
    try:
        model = genai.GenerativeModel('gemini-1.5-pro')
        prompt = f"""
        You are a mental health support assistant. Based on the user's message, emotional state, and provided tip, 
        respond with empathy and incorporate the tip. Keep the response concise, ~2 lines, under 50 words.

        Examples:
        1. User: "I feel overwhelmed with deadlines."
           Emotion: Stressed
           Tip: Take slow, deep breaths
           Response: Deadlines are tough! Try slow, deep breaths to find calm.

        2. User: "I just feel really low today."
           Emotion: Sad
           Tip: Write down three things you’re grateful for
           Response: Feeling low is hard. Writing three things you’re grateful for may help.

        3. User: "I keep worrying about everything."
           Emotion: Anxious
           Tip: Practice grounding: name 5 things you see
           Response: Worrying feels heavy! Try naming 5 things you see to ground yourself.

        Now respond to:
        User: {transcript}
        Emotion: {emotion}
        Tip: {tip}
        Response:
        """
        response = generate_content_with_retry(
            model,
            prompt,
            generation_config={"max_output_tokens": 60, "temperature": 0.5}
        )
        return response.text.strip() if response.text else "Sorry, I couldn’t generate a response. Try again."
    except Exception as e:
        return f"Response generation failed: {str(e)}"

### Standalone Test

In [10]:
test_transcript = "Work is driving me crazy"
test_emotion = "Stressed"
test_tip = retrieve_tip(test_emotion)  # Assumes retrieve_tip from Section 4
response = generate_empathetic_response_fewshot(test_transcript, test_emotion, test_tip)
print(f"Transcript: {test_transcript}")
print(f"Emotion: {test_emotion}")
print(f"Tip: {test_tip}")
print(f"Response: {response}\n")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Transcript: Work is driving me crazy
Emotion: Stressed
Tip: Describe a view outside.
Response: Work stress is so draining. Taking a moment to describe the view outside might offer a helpful mental break.



# 6. Agents: Conversation Flow

Using Day 3’s agent design and Day 5’s governance, we orchestrate the pipeline, displaying responses and escalating after 3 negative turns to prioritize user safety.

**Output**:
> Agent Output: I hear you… Try deep breaths + helpline if needed.

**Goal**: Ensure ethical, supportive interactions.

In [11]:
class ConversationAgent:
    """Manages conversation flow, tracks negative sentiments, and escalates if needed."""
    def __init__(self):
        self.negative_count = 0
        self.helpline_message = (
            "It sounds like you're really struggling. Consider reaching out to a helpline like "
            "988 (US) or a local mental health service for support."
        )

    def process_interaction(self, transcript):
        """Process one interaction, return response, and handle escalation."""
        # Detect emotion and sentiment
        emotion, sentiment, error = detect_emotion(transcript)
        if error:
            return f"Error: {error}"

        # Track negative sentiments
        if sentiment == "Negative":
            self.negative_count += 1
        else:
            self.negative_count = 0  # Reset on positive sentiment

        # Retrieve tip
        tip = retrieve_tip(emotion)

        # Generate response
        response = generate_empathetic_response_fewshot(transcript, emotion, tip)

        # Escalate after 3 negative turns
        if self.negative_count >= 3:
            response += f" {self.helpline_message}"
            self.negative_count = 0  # Reset after escalation

        return {
            "Transcript": transcript,
            "Category": emotion,
            "Sentiment": sentiment,
            "Tip": tip,
            "Agent Output": response
        }

## Test Agent

In [12]:
agent = ConversationAgent()
test_transcripts = [
    "Work is driving me crazy",  # Negative
    "I feel so empty today",    # Negative
    "I can’t keep up anymore"   # Negative (should trigger escalation)
]
for transcript in test_transcripts:
    result = agent.process_interaction(transcript)
    for key, value in result.items():
        print(f"{key}: {value}")
    print("-" * 50)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Transcript: Work is driving me crazy
Category: Frustrated
Sentiment: Negative
Tip: Look at a distant point.
Agent Output: Work frustration is so draining. Try looking at a distant point to ease your mind.
--------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Transcript: I feel so empty today
Category: Sad
Sentiment: Negative
Tip: Allow yourself to feel and cry.
Agent Output: It's okay to feel empty.  Allowing yourself to feel and even cry can be helpful.
--------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Transcript: I can’t keep up anymore
Category: Stressed
Sentiment: Negative
Tip: Describe a view outside.
Agent Output: It's okay to feel overwhelmed.  Describing the view outside might bring a moment of peace. It sounds like you're really struggling. Consider reaching out to a helpline like 988 (US) or a local mental health service for support.
--------------------------------------------------


# 7. Full Interaction Demo

This end-to-end flow ties Days 1-5 into a seamless, voice-driven experience that Kaggle viewers can run and love.

**Output**:
>Transcript: I’m sad

>Category: Sad, Sentiment: Negative

>Tip: Try a short walk to lift your mood.

>Response: I’m here for you—feeling sad can be heavy. Try a short walk to lift your mood.

**Goal**: Showcase innovation and empathy in action.

In [13]:
def full_interaction(audio_file, agent):
    """Run full pipeline with agent for conversation flow."""
    # Transcribe audio
    transcript = transcribe_audio(audio_file)
    # Process via agent
    result = agent.process_interaction(transcript)
    return result

# Demo with multiple clips
agent = ConversationAgent()
audio_files = [
    "/kaggle/input/emotions-speech/angry_2.wav",
    "/kaggle/input/emotions-speech/sad_1.wav"
]
for audio_file in audio_files:
    result = full_interaction(audio_file, agent)
    for key, value in result.items():
        print(f"{key}: {value}")
    print("-" * 50)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Transcript: why does this keep happening
Category: Frustrated
Sentiment: Negative
Tip: Look at a distant point.
Agent Output: It's frustrating when things feel repeated. Try looking at something far away to shift your focus.
--------------------------------------------------


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Transcript: I feel so empty today
Category: Sad
Sentiment: Negative
Tip: Allow yourself to feel and cry.
Agent Output: It's okay to feel empty. Allowing yourself to feel, even crying, can be a healthy release.
--------------------------------------------------


# 8. Evaluation and Governance

**🎯 Goal:**  
To critically assess TheraBot’s performance, ethical integrity, and readiness to offer scalable mental health support—while ensuring safety, transparency, and **responsible AI use.

### ✅ **Strengths**

- **Voice-Driven Emotional Intelligence**  
  TheraBot transcribes and interprets voice inputs to detect emotional states like Stressed, Sad, or Anxious with contextual precision.

- **Empathetic Response Generation**  
  Gemini’s few-shot prompting, paired with RAG, enables context-aware, emotionally intelligent responses grounded in evidence-based resources.

- **Escalation Logic**  
  Built-in agent logic triggers helpline prompts after detecting repeated distress, ensuring TheraBot remains proactive, not passive.

- **Synthetic Training Data**  
  Trained entirely on ethically crafted synthetic datasets (text-to-speech + emotion tags), reducing privacy risks and avoiding real user data exposure.

- **Governance Embedded by Design**  
  A responsible-by-default framework with curated content, filtered outputs, and escalation triggers aligned to safety protocols.

### ⚠️ **Limitations**

- **Audio Accuracy Gaps**  
  Performance can decline with low-quality microphones, background noise, or strong regional accents.

- **Synthetic Bias Boundaries**  
  While safer, synthetic data can limit emotional nuance. Testing with diverse, anonymized real-world samples is advised.

- **Tip Base Scalability**  
  The current tip database (50–200 items) is limited. Scalability demands a larger, continuously updated knowledge source.

- **Therapeutic Boundaries**  
  TheraBot is not a diagnostic or clinical tool. It provides support, but does not replace professional mental health care.

### 🔐 **Governance Highlights**

- **Sourced Tips from Trusted Institutions**  
  Tips are pulled from credible sources like WHO, NIH, etc., and manually reviewed to avoid clinical prescriptions.

- **Bias & Harm Filtering**  
  Responses are filtered to prevent harmful content, with escalation logic activating after three distress signals.

- **Built-in Disclaimers**  
  Every interaction clarifies that TheraBot is a companion, not a medical advisor.

- **Ethics at the Core**  
  Designed to listen, support, and redirect—ensuring value delivery while maintaining user trust and ethical integrity.

# 9. Conclusion

TheraBot isn't just a tech demo—it's a call to build AI that puts people first. By combining emotional insight, generative intelligence, and strict governance, we've created a system that listens, responds with care, and acts responsibly.

We don’t claim to solve mental health. But we bridge the first mile of support—with empathy, safety, and respect. With further iteration, real-world validation, and stronger partnerships, TheraBot can become a frontline AI ally for mental wellness.